# Bike Regression — ทำนายจำนวนจักรยานที่ถูกเช่า

In [ ]:
import sys
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
sys.path.insert(0, str(REPO))
import config
print("RANDOM_STATE =", config.RANDOM_STATE)

RANDOM_STATE = 42


## 1. ปัญหา / เป้าหมาย
**ทำนายอะไร:** จำนวนจักรยานสาธารณะที่ถูกเช่าในเมืองโซลใน 1 ชั่วโมง (`Rented Bike Count`) **หน่วย: คัน/ชั่วโมง** · เป็นงาน Regression เพราะค่าที่ทำนายเป็นตัวเลขต่อเนื่อง

**ตัวแปรนำเข้า:** ข้อมูลเวลา (วันที่, ชั่วโมง, วันหยุด) และสภาพอากาศ (อุณหภูมิ, ความชื้น, ความเร็วลม, ปริมาณฝน) · input ที่ใช้จริงในแอปจะเลือกและให้เหตุผลในหัวข้อ 5

**ประโยชน์:** ผู้ให้บริการรู้ล่วงหน้าว่าชั่วโมงไหนคนจะเช่ามากหรือน้อย จึงใช้ได้กับ
- เตรียมจักรยานให้พอในชั่วโมงที่ความต้องการสูง ไม่ให้ผู้ใช้หาจักรยานไม่ได้
- จัดคนงานย้ายจักรยานไปจุดต่างๆ ก่อนช่วงที่คนเช่าเยอะ
- เลือกช่วงที่ความต้องการต่ำไว้ซ่อมบำรุง


## 2. แหล่งข้อมูล
- **ชุดข้อมูล:** Seoul Bike Sharing Demand จาก [UCI Machine Learning Repository (id 560)](https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand) · DOI: 10.24432/C5F62R
- **ไลเซนส์:** CC BY 4.0 นำไปใช้และเผยแพร่ต่อได้ โดยต้องอ้างอิงที่มา
- **ไฟล์:** `bike-regression/data/SeoulBikeData.csv`
- **ขนาด:** 8,760 แถว × 14 คอลัมน์ · 1 แถว = 1 ชั่วโมงของทั้งระบบ (ยอดรวมทุกสถานี) ตั้งแต่ 1 ธ.ค. 2017 ถึง 30 พ.ย. 2018

**คอลัมน์**
- **เป้าหมาย:** `Rented Bike Count` จำนวนจักรยานที่ถูกเช่าในชั่วโมงนั้น (คัน)
- **เวลา:** `Date` (วันที่), `Hour` (0–23), `Seasons` (ฤดู), `Holiday` (วันหยุดหรือไม่), `Functioning Day` (ระบบเปิดให้บริการหรือไม่)
- **สภาพอากาศ:** `Temperature(°C)`, `Humidity(%)`, `Wind speed (m/s)`, `Visibility (10m)`, `Dew point temperature(°C)`, `Solar Radiation (MJ/m2)`, `Rainfall(mm)`, `Snowfall (cm)`

## 3. โหลดและดูข้อมูล
- ดูโครงสร้างของข้อมูลทั้งชุด: encoding, ตารางตัวอย่าง, ชนิดข้อมูล, ค่าที่หายไป, ค่าในคอลัมน์หมวดหมู่

### 3.1 ตรวจ encoding และรูปแบบวันที่ของไฟล์ดิบ
**จุดประสงค์**
1. หา encoding ที่อ่านไฟล์ได้ถูกต้อง เพราะชื่อคอลัมน์มีสัญลักษณ์ `°` (เช่น `Temperature(°C)`) ถ้าเลือก encoding ผิด จะอ่านไฟล์ไม่ได้
2. หารูปแบบของคอลัมน์ `Date` (เช่น `01/12/2017`) ว่าเป็น วัน/เดือน หรือ เดือน/วัน ถ้าเลือกผิด วันที่จะสลับกัน และการแบ่งข้อมูลตามวันในขั้นถัดไปจะผิดตาม

In [ ]:
import pandas as pd

# (ก) ดู byte ดิบของหัวตาราง ตรงคอลัมน์อุณหภูมิ
with open(config.BIKE_RAW_CSV, "rb") as f:
    header_bytes = f.readline()
i = header_bytes.find(b"Temperature")
print("bytes ของหัวคอลัมน์อุณหภูมิ:", header_bytes[i:i + 16])

# (ข) ลองอ่านหัวตารางด้วย 2 encoding
for enc in ["utf-8", "cp1252"]:
    try:
        cols = pd.read_csv(config.BIKE_RAW_CSV, encoding=enc, nrows=0).columns
        print(f"{enc} -> อ่านได้:", cols[3])
    except UnicodeDecodeError as e:
        print(f"{enc} -> อ่านไม่ได้:", e.reason)

# (ค) ตรวจรูปแบบวันที่: แยก "01/12/2017" เป็น 3 ส่วน แล้วดูค่าสูงสุดของแต่ละส่วน
dates = pd.read_csv(config.BIKE_RAW_CSV, encoding="cp1252", usecols=["Date"])["Date"] 
parts = dates.str.split("/", expand=True).astype(int)
print("ส่วนที่ 1 ค่าสูงสุด:", parts[0].max())
print("ส่วนที่ 2 ค่าสูงสุด:", parts[1].max())
print("ส่วนที่ 3 ค่าที่มี:", sorted(parts[2].unique().tolist()))

# (ง) แปลงเป็นวันที่ด้วยรูปแบบ วัน/เดือน/ปี แล้วตรวจช่วงวันที่และจำนวนวัน
parsed = pd.to_datetime(dates, format="%d/%m/%Y")
print("ช่วงวันที่:", parsed.min().date(), "ถึง", parsed.max().date())
print("จำนวนวัน:", parsed.nunique(), "| จำนวนแถว:", len(parsed))

bytes ของหัวคอลัมน์อุณหภูมิ: b'Temperature(\xb0C),'
utf-8 -> อ่านไม่ได้: invalid start byte
cp1252 -> อ่านได้: Temperature(°C)
ส่วนที่ 1 ค่าสูงสุด: 31
ส่วนที่ 2 ค่าสูงสุด: 12
ส่วนที่ 3 ค่าที่มี: [2017, 2018]
ช่วงวันที่: 2017-12-01 ถึง 2018-11-30
จำนวนวัน: 365 | จำนวนแถว: 8760


**ผลลัพธ์**
- `°` ถูกเก็บเป็น byte `\xb0`: อ่านด้วย `utf-8` ไม่ได้ แต่อ่านด้วย `cp1252` ได้ถูกต้อง → ใช้ `cp1252`
- วันที่ส่วนแรกมีค่าถึง 31 ส่วนที่สองไม่เกิน 12 → รูปแบบเป็น วัน/เดือน/ปี (`%d/%m/%Y`)
- ข้อมูลตั้งแต่ 1 ธ.ค. 2017 ถึง 30 พ.ย. 2018 มี 365 วัน × 24 ชม. = 8,760 แถว ไม่มีชั่วโมงไหนหาย
- บันทึกทั้งสองค่าไว้ใน `config.py`

### 3.2 โหลดข้อมูลและเปลี่ยนชื่อคอลัมน์
**จุดประสงค์**
- อ่านไฟล์ด้วย encoding ที่ตรวจแล้วใน 3.1 (`cp1252`)
- เปลี่ยนชื่อคอลัมน์เป็นแบบสั้นที่ไม่มีหน่วยหรืออักขระพิเศษ (เช่น `Temperature(°C)` → `temp_c`) เพื่อลดความผิดพลาดตอนเขียนโค้ด
- แปลงคอลัมน์ `date` จากข้อความเป็นวันที่จริง ด้วยรูปแบบที่ตรวจแล้วใน 3.1 (`%d/%m/%Y` = วัน/เดือน/ปี)

In [ ]:
# อ่าน CSV ทั้งไฟล์ด้วย encoding ที่ตรวจแล้วใน 3.1 (cp1252)
df = pd.read_csv(config.BIKE_RAW_CSV, encoding=config.BIKE_RAW_ENCODING)

# ตรวจว่าชื่อคอลัมน์ในไฟล์ตรงกับ config ทุกตัวและทุกลำดับ (ไฟล์ถูกเปลี่ยน → หยุดทันที)
assert list(df.columns) == list(config.BIKE_COLUMN_MAP), "ชื่อคอลัมน์ไม่ตรงกับ config.BIKE_COLUMN_MAP"

# เปลี่ยนชื่อคอลัมน์เป็นแบบสั้น เช่น "Temperature(°C)" → "temp_c"
df = df.rename(columns=config.BIKE_COLUMN_MAP)

# แปลง date จากข้อความ (object) เป็นวันที่ (datetime64) ด้วยรูปแบบ วัน/เดือน/ปี
df[config.BIKE_DATE_COL] = pd.to_datetime(df[config.BIKE_DATE_COL], format=config.BIKE_DATE_FORMAT)

# แสดงขนาดตาราง (แถว, คอลัมน์) และ 5 แถวแรก
print("shape:", df.shape)
df.head()

shape: (8760, 14)


,date,rented_bike_count,hour,temp_c,humidity_pct,wind_speed_ms,visibility_10m,dew_point_c,solar_radiation_mj,rainfall_mm,snowfall_cm,seasons,holiday,functioning_day
0,2017-12-01,254,0,-5.2,37,2.2,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
1,2017-12-01,204,1,-5.5,38,0.8,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
2,2017-12-01,173,2,-6.0,39,1.0,2000,-17.7,0.0,0.0,0.0,Winter,No Holiday,Yes
3,2017-12-01,107,3,-6.2,40,0.9,2000,-17.6,0.0,0.0,0.0,Winter,No Holiday,Yes
4,2017-12-01,78,4,-6.0,36,2.3,2000,-18.6,0.0,0.0,0.0,Winter,No Holiday,Yes


**ผลลัพธ์:** ได้ 8,760 แถว × 14 คอลัมน์ ชื่อคอลัมน์ตรงกับที่คาดทุกตัว · แถวแรกคือ 1 ธ.ค. 2017 เวลา 0:00 ยอดเช่า 254 คัน อุณหภูมิ −5.2 °C

### 3.3 ชนิดข้อมูลและความครบถ้วนของข้อมูล
**จุดประสงค์**
- ดูชนิดข้อมูลของแต่ละคอลัมน์ ว่าเป็นตัวเลข ข้อความ หรือวันที่ เพื่อรู้ว่าคอลัมน์ไหนต้องแปลงก่อนเข้าโมเดล
- ตรวจ missing value และแถวซ้ำ
- ตรวจว่าทุกวันมีครบ 24 ชั่วโมง (0–23) เพราะการแบ่งข้อมูลในหัวข้อ 4 ใช้ "วัน" เป็นหน่วย

In [ ]:
# ชนิดข้อมูล (Dtype) และจำนวนค่าที่ไม่ว่าง (Non-Null) ของทุกคอลัมน์
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   date                8760 non-null   datetime64[ns]
 1   rented_bike_count   8760 non-null   int64         
 2   hour                8760 non-null   int64         
 3   temp_c              8760 non-null   float64       
 4   humidity_pct        8760 non-null   int64         
 5   wind_speed_ms       8760 non-null   float64       
 6   visibility_10m      8760 non-null   int64         
 7   dew_point_c         8760 non-null   float64       
 8   solar_radiation_mj  8760 non-null   float64       
 9   rainfall_mm         8760 non-null   float64       
 10  snowfall_cm         8760 non-null   float64       
 11  seasons             8760 non-null   object        
 12  holiday             8760 non-null   object        
 13  functioning_day     8760 non-null   object      

In [ ]:
rows_per_day = df.groupby("date").size()  # นับจำนวนแถวของแต่ละวัน
hours_complete = (                        # เก็บผลว่าทุกวันมีชั่วโมงครบไหม (True/False)
    df.groupby("date")["hour"]            # จัดกลุ่มคอลัมน์ hour ตามวัน
    .apply(lambda h: sorted(h) == list(range(24)))  # แต่ละวัน: ชั่วโมงที่เรียงแล้วต้องเท่ากับ [0, 1, …, 23]
    .all()                                # ทุกวันต้องเป็น True จึงได้ True
)

checks = pd.Series({                                                       # รวมผลตรวจทุกข้อเป็นตารางเดียว
    "จำนวนแถว": len(df),                                                    # จำนวนแถวทั้งหมด
    "missing (รวมทุกคอลัมน์)": int(df.isna().sum().sum()),                   # นับ missing ต่อคอลัมน์ แล้วรวมทุกคอลัมน์
    "แถวซ้ำทั้งแถว": int(df.duplicated().sum()),                              # แถวที่ทุกคอลัมน์ซ้ำกับแถวก่อนหน้า
    "แถวซ้ำ (date, hour)": int(df.duplicated(["date", "hour"]).sum()),       # วันและชั่วโมงเดียวกันมีมากกว่า 1 แถว
    "วันแรก": df["date"].min().date(),                                       # วันที่เก่าที่สุด
    "วันสุดท้าย": df["date"].max().date(),                                   # วันที่ใหม่ที่สุด
    "จำนวนวัน": df["date"].nunique(),                                        # จำนวนวันที่ไม่ซ้ำกัน
    "แถวต่อวัน (min–max)": f"{rows_per_day.min()}–{rows_per_day.max()}",    # วันที่มีแถวน้อยสุดและมากสุด
    "ทุกวันมีชั่วโมง 0–23 ครบ": hours_complete,                               # ผลจากด้านบน
}, dtype=object)                                                           # object = ใส่ตัวเลข วันที่ ข้อความปนกันได้

# ขนาดข้อมูลต้องถูกต้อง: 8,760 แถว = 365 วัน × 24 ชม. และทุกวันมีชั่วโมง 0–23 ครบ
assert len(df) == 8760 and df["date"].nunique() == 365 and hours_complete
# ข้อมูลต้องสะอาด: ไม่มี missing และไม่มีชั่วโมงไหนซ้ำในวันเดียวกัน
assert df.isna().sum().sum() == 0 and df.duplicated(["date", "hour"]).sum() == 0
checks.to_frame("ค่า")  # แสดงผลเป็นตาราง 1 คอลัมน์ชื่อ "ค่า"

,ค่า
จำนวนแถว,8760
missing (รวมทุกคอลัมน์),0
แถวซ้ำทั้งแถว,0
"แถวซ้ำ (date, hour)",0
วันแรก,2017-12-01
วันสุดท้าย,2018-11-30
จำนวนวัน,365
แถวต่อวัน (min–max),24–24
ทุกวันมีชั่วโมง 0–23 ครบ,True


**ผลลัพธ์**
- `date` เป็น `datetime64` แล้ว · สภาพอากาศเป็นตัวเลข (int/float) · `seasons`, `holiday`, `functioning_day` เป็นข้อความ (`object`) → ต้องแปลงเป็นตัวเลขก่อนเข้าโมเดล
- **ไม่มี missing value** และไม่มีแถวซ้ำ
- 365 วัน ทุกวันมี 24 แถว ชั่วโมง 0–23 ครบ → ใช้ "วัน" เป็นหน่วยของการแบ่งข้อมูลได้

## 4. แบ่งข้อมูล
**จุดประสงค์:** กันข้อมูลส่วนหนึ่ง (test) ไว้วัดผลตอนท้ายครั้งเดียว โมเดลต้องไม่เคยเห็นข้อมูลส่วนนี้ตอนฝึกหรือตอนเลือกโมเดล

**วิธีแบ่ง:** สุ่มแบ่ง **ตามวัน** ไม่สุ่มทีละแถว
- 1 วันมี 24 แถว และชั่วโมงติดกันมีอากาศกับยอดเช่าใกล้กันมาก ถ้าชั่วโมงหนึ่งอยู่ train แต่ชั่วโมงถัดไปอยู่ test โมเดลจะแค่จำวันนั้นได้ คะแนน test จะสูงเกินจริง (leakage)
- ใช้ `GroupShuffleSplit(test_size=0.2, random_state=42)` โดยให้ `groups` เป็นวันที่ ทั้ง 24 ชม. ของวันเดียวกันจึงไปอยู่ฝั่งเดียวกันเสมอ
- เรียงข้อมูลตาม (วัน, ชั่วโมง) ก่อนสุ่ม รันกี่ครั้งก็ได้ผลเดิม
- บันทึกรายการวันไว้ที่ `bike-regression/data/splits/` เพื่อให้ทุกส่วนใช้วันชุดเดียวกัน

## 5. EDA และเตรียมข้อมูล (train เท่านั้น)
- TODO: distribution ของยอดเช่า, ความสัมพันธ์กับชั่วโมง/ฤดู/อุณหภูมิ/ฝน, correlation, outlier
- TODO: ตัดชั่วโมงที่ระบบปิด + สร้าง feature จากวันที่และชั่วโมง
- TODO: เลือก input ของแอป **≤ 8 feature** + เหตุผล (ผู้ใช้กรอกได้จริง)
- ⚠️ ตัดสินใจจาก train เท่านั้น

In [ ]:
# TODO

## 6. Train + เหตุผล
- TODO: preprocessing + model ใน `Pipeline` เดียว
- TODO: baseline + เทียบ 4 โมเดลด้วย `GroupKFold` (group = วันที่) บน train แล้ว tune ตัวที่ดีที่สุด

In [ ]:
# TODO

## 7. ประเมินบน test + แปลความหมาย
- เมตริก: **MAE (หลัก, คัน/ชม.)**, RMSE, R²
- ⚠️ MAPE ใช้ตรงๆ ไม่ได้ เพราะมียอด = 0
- TODO: แปลความหมาย MAE เทียบกับยอดเช่าเฉลี่ย

In [ ]:
# TODO

## 8. ตัวอย่างผลทำนาย + error analysis
- TODO: ตัวอย่างทำนาย 3–5 แถว · residual vs ชั่วโมง/ฤดู · ช่วงที่ผิดมาก

In [ ]:
# TODO

## 9. บันทึกโมเดล → `app/model.joblib`
- TODO: `joblib.dump(pipeline, config.BIKE_MODEL)` — ทั้ง Pipeline ไฟล์เดียว
- ⚠️ เช็กขนาดไฟล์ < 100 MB · ลองโหลดกลับแล้วทำนายซ้ำให้ได้ค่าเดิม

In [ ]:
# TODO